In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
df= pd.read_csv('/kaggle/input/datasets/rhythmghai/netflix-user-watching-behavior-dataset/netflix_user_behavior_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/rhythmghai/netflix-user-watching-behavior-dataset/netflix_user_behavior_dataset.csv'

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x="churned", data=df)

In [ ]:
sns.boxplot(x="churned", y="account_age_months", data=df)

In [ ]:
df.info()

In [ ]:

X = df.drop(["churned", "user_id"], axis=1)
y = df["churned"]


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), make_column_selector(dtype_include=['int64', 'float64'])),
    ("cat", OneHotEncoder(handle_unknown='ignore'), make_column_selector(dtype_include='object'))
])

In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report


In [ ]:

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()


xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.03,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)


pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("model", xgb_model)
])


pipe.fit(X_train, y_train)


y_pred = pipe.predict(X_test)

In [ ]:
acc = accuracy_score(y_test, y_pred)

print("XGBOOST RESULTS")
print("Accuracy:", acc)
print("Classification Report:\n", 
      classification_report(y_test, y_pred, zero_division=0))

In [1]:
import pickle